In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv("../Dataset/weekly_student_dataset.csv")

print(df.head())

print(df.shape)

print(df.columns.tolist())

   id_student code_module code_presentation  week  weekly_video_clicks  \
0        6516         AAA             2014J     1                  151   
1        6516         AAA             2014J     2                    9   
2        6516         AAA             2014J     3                   31   
3        6516         AAA             2014J     4                  241   
4        6516         AAA             2014J     5                   56   

   weekly_login_frequency  weekly_avg_activity_day  weekly_avg_quiz_score  \
0                       3               -21.280000                    0.0   
1                       2               -15.333333                    0.0   
2                       2                -5.333333                    0.0   
3                       5                -0.081081                    0.0   
4                       4                 5.333333                   60.0   

   weekly_assessments_completed  weekly_avg_submission_day  \
0                           0.

In [5]:
print(df.isnull().sum())

df = df.fillna(0)

print("Missing Values Removed")

id_student                      0
code_module                     0
code_presentation               0
week                            0
weekly_video_clicks             0
weekly_login_frequency          0
weekly_avg_activity_day         0
weekly_avg_quiz_score           0
weekly_assessments_completed    0
weekly_avg_submission_day       0
weekly_avg_assessment_weight    0
dropout                         0
dtype: int64
Missing Values Removed


In [6]:
label_encoder = LabelEncoder()

categorical_columns = [
    "code_module",
    "code_presentation"
]

for column in categorical_columns:

    df[column] = label_encoder.fit_transform(df[column])

print(df.head())

   id_student  code_module  code_presentation  week  weekly_video_clicks  \
0        6516            0                  3     1                  151   
1        6516            0                  3     2                    9   
2        6516            0                  3     3                   31   
3        6516            0                  3     4                  241   
4        6516            0                  3     5                   56   

   weekly_login_frequency  weekly_avg_activity_day  weekly_avg_quiz_score  \
0                       3               -21.280000                    0.0   
1                       2               -15.333333                    0.0   
2                       2                -5.333333                    0.0   
3                       5                -0.081081                    0.0   
4                       4                 5.333333                   60.0   

   weekly_assessments_completed  weekly_avg_submission_day  \
0                 

In [7]:
features = [

    "weekly_video_clicks",

    "weekly_login_frequency",

    "weekly_avg_activity_day",

    "weekly_avg_quiz_score",

    "weekly_assessments_completed",

    "weekly_avg_submission_day",

    "weekly_avg_assessment_weight",

    "code_module",

    "code_presentation"

]

target = "dropout"

In [8]:
scaler = StandardScaler()

df[features] = scaler.fit_transform(df[features])

print(df.head())

   id_student  code_module  code_presentation  week  weekly_video_clicks  \
0        6516     -1.85866           1.129025     1             0.921143   
1        6516     -1.85866           1.129025     2            -0.562918   
2        6516     -1.85866           1.129025     3            -0.332993   
3        6516     -1.85866           1.129025     4             1.861745   
4        6516     -1.85866           1.129025     5            -0.071715   

   weekly_login_frequency  weekly_avg_activity_day  weekly_avg_quiz_score  \
0                0.072551                -1.611406              -0.464081   
1               -0.485020                -1.533861              -0.464081   
2               -0.485020                -1.403461              -0.464081   
3                1.187695                -1.334972              -0.464081   
4                0.630123                -1.264368               1.476099   

   weekly_assessments_completed  weekly_avg_submission_day  \
0                 

In [9]:
sequence_length = 4

X = []

y = []

In [10]:
students = df["id_student"].unique()

for student in students:

    student_data = df[df["id_student"] == student]

    student_data = student_data.sort_values("week")

    values = student_data[features].values

    labels = student_data[target].values

    if len(values) < sequence_length + 1:
        continue

    for i in range(len(values) - sequence_length):

        X.append(
            values[i:i+sequence_length]
        )

        y.append(
            labels[i+sequence_length]
        )

X = np.array(X)

y = np.array(y)

print(X.shape)

print(y.shape)

(530025, 4, 9)
(530025,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print(X_train.shape)

print(X_test.shape)

(424020, 4, 9)
(106005, 4, 9)


In [12]:
class_weights = compute_class_weight(

    class_weight="balanced",

    classes=np.unique(y_train),

    y=y_train

)

class_weights = {

    0: class_weights[0],

    1: class_weights[1]

}

print(class_weights)

{0: np.float64(0.5462035491250851), 1: np.float64(5.910839745734359)}


In [13]:
model = Sequential()

model.add(

    LSTM(

        64,

        input_shape=(

            X_train.shape[1],

            X_train.shape[2]

        )

    )

)

model.add(

    Dropout(0.3)

)

model.add(

    Dense(

        32,

        activation="relu"

    )

)

model.add(

    Dropout(0.2)

)

model.add(

    Dense(

        1,

        activation="sigmoid"

    )

)

In [14]:
model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]

)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,057 (82.25 KB)

 Trainable params: 21,057 (82.25 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True

)

In [16]:
history = model.fit(

    X_train,

    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=32,

    class_weight=class_weights,

    callbacks=[early_stop],

    verbose=1

)

Epoch 1/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 46s 4ms/step - accuracy: 0.6550 - loss: 0.5472 - val_accuracy: 0.6921 - val_loss: 0.5406
Epoch 2/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 40s 4ms/step - accuracy: 0.6865 - loss: 0.5265 - val_accuracy: 0.7297 - val_loss: 0.4727
Epoch 3/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - accuracy: 0.7042 - loss: 0.5153 - val_accuracy: 0.6909 - val_loss: 0.5341
Epoch 4/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - accuracy: 0.7094 - loss: 0.5075 - val_accuracy: 0.7083 - val_loss: 0.5199
Epoch 5/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 38s 4ms/step - accuracy: 0.7138 - loss: 0.5005 - val_accuracy: 0.7398 - val_loss: 0.4882


In [17]:
y_probability = model.predict(X_test).flatten()

print(y_probability[:10])

3313/3313 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
[0.34770656 0.7401526  0.26113892 0.06160431 0.87514883 0.5761868
 0.02446733 0.1343674  0.367642   0.01097725]


In [18]:
from sklearn.metrics import f1_score

thresholds = np.arange(

    0.10,

    0.91,

    0.05

)

best_threshold = 0.5

best_f1 = 0

for t in thresholds:

    y_pred = (

        y_probability >= t

    ).astype(int)

    score = f1_score(

        y_test,

        y_pred

    )

    print(

        f"Threshold={t:.2f}  F1={score:.4f}"

    )

    if score > best_f1:

        best_f1 = score

        best_threshold = t

print()

print("Best Threshold :", best_threshold)

print("Best F1 Score :", best_f1)

Threshold=0.10  F1=0.2104
Threshold=0.15  F1=0.2250
Threshold=0.20  F1=0.2379
Threshold=0.25  F1=0.2508
Threshold=0.30  F1=0.2623
Threshold=0.35  F1=0.2756
Threshold=0.40  F1=0.2888
Threshold=0.45  F1=0.3029
Threshold=0.50  F1=0.3175
Threshold=0.55  F1=0.3331
Threshold=0.60  F1=0.3491
Threshold=0.65  F1=0.3592
Threshold=0.70  F1=0.3607
Threshold=0.75  F1=0.3457
Threshold=0.80  F1=0.2966
Threshold=0.85  F1=0.2147
Threshold=0.90  F1=0.0998

Best Threshold : 0.7000000000000002
Best F1 Score : 0.36073713358159076


In [19]:
y_prediction = (

    y_probability >= best_threshold

).astype(int)

In [20]:
print(

    "Accuracy :",

    accuracy_score(

        y_test,

        y_prediction

    )

)

print(

    "Precision :",

    precision_score(

        y_test,

        y_prediction

    )

)

print(

    "Recall :",

    recall_score(

        y_test,

        y_prediction

    )

)

print(

    "F1 Score :",

    f1_score(

        y_test,

        y_prediction

    )

)

print(

    "ROC AUC :",

    roc_auc_score(

        y_test,

        y_probability

    )

)

print("\nConfusion Matrix")

print(

    confusion_matrix(

        y_test,

        y_prediction

    )

)

print("\nClassification Report")

print(

    classification_report(

        y_test,

        y_prediction

    )

)

Accuracy : 0.8736852035281355
Precision : 0.31538525753401786
Recall : 0.4213226274116204
F1 Score : 0.36073713358159076
ROC AUC : 0.8179977420546424

Confusion Matrix
[[88837  8201]
 [ 5189  3778]]

Classification Report
              precision    recall  f1-score   support

           0       0.94      0.92      0.93     97038
           1       0.32      0.42      0.36      8967

    accuracy                           0.87    106005
   macro avg       0.63      0.67      0.65    106005
weighted avg       0.89      0.87      0.88    106005



In [21]:
future_risk = []

for prob in y_probability:

    if prob < 0.20:

        future_risk.append("Low")

    elif prob < 0.50:

        future_risk.append("Medium")

    else:

        future_risk.append("High")

In [22]:
intervention = []

for risk in future_risk:

    if risk == "Low":

        intervention.append(
            "Continue Current Learning Path"
        )

    elif risk == "Medium":

        intervention.append(
            "Weekly Mentor Follow-up"
        )

    else:

        intervention.append(
            "Immediate Faculty Intervention"
        )

In [23]:
future_prediction = pd.DataFrame({

    "Actual": y_test,

    "Predicted": y_prediction,

    "Risk Probability": y_probability,

    "Future Risk": future_risk,

    "Recommended Intervention": intervention

})

future_prediction.head()

,Actual,Predicted,Risk Probability,Future Risk,Recommended Intervention
0,0,0,0.347707,Medium,Weekly Mentor Follow-up
1,1,1,0.740153,High,Immediate Faculty Intervention
2,0,0,0.261139,Medium,Weekly Mentor Follow-up
3,0,0,0.061604,Low,Continue Current Learning Path
4,0,1,0.875149,High,Immediate Faculty Intervention


In [24]:
future_prediction.to_csv(

    "../Dataset/future_risk_prediction.csv",

    index=False

)

print(

    "Future Risk Prediction Dataset Saved Successfully"

)

Future Risk Prediction Dataset Saved Successfully


In [25]:
print(

    future_prediction["Future Risk"].value_counts()

)

Future Risk
Low       43784
High      32881
Medium    29340
Name: count, dtype: int64


In [29]:
model.save(

    "../Models/future_risk_lstm_model.keras"

)

print(

    "LSTM Model Saved Successfully"

)

LSTM Model Saved Successfully


In [30]:
import joblib

joblib.dump(

    scaler,

    "../Models/lstm_scaler.pkl"

)

print(

    "Scaler Saved Successfully"

)

Scaler Saved Successfully


In [31]:
print("="*60)

print("Notebook 8 Completed Successfully")

print("="*60)

print("Model           : LSTM")

print("Sequence Length :", sequence_length)

print("Best Threshold  :", best_threshold)

print("ROC AUC         :", roc_auc_score(y_test, y_probability))

print("Prediction File : future_risk_prediction.csv")

print("Saved Model     : future_risk_lstm_model.keras")

print("="*60)

Notebook 8 Completed Successfully
Model           : LSTM
Sequence Length : 4
Best Threshold  : 0.7000000000000002
ROC AUC         : 0.8179977420546424
Prediction File : future_risk_prediction.csv
Saved Model     : future_risk_lstm_model.keras
